# CardioCCC NER Pipeline — lcampillos/roberta-es-clinical-trials-ner

This notebook presents a complete workflow for named entity recognition in clinical texts.
It brings together configuration, training on DisTEMIST, inference and external evaluation on CardioCCC.

## Contents

1. [Environment and dependencies](#1-environment-and-dependencies)
2. [Configuration](#2-configuration)
   - [Paths and dataset](#21-paths-and-dataset)
   - [Labels and data loading](#22-labels-and-data-loading)
   - [Segmentation and label alignment](#23-segmentation-and-label-alignment)
   - [Hyperparameters and tokenizer](#24-hyperparameters-and-tokenizer)
3. [Training](#3-training)
   - [Evaluation metrics](#31-evaluation-metrics)
   - [Discriminative fine-tuning](#32-discriminative-fine-tuning)
   - [Multi-seed k-fold loop](#33-multi-seed-k-fold-loop)
   - [Ensemble summary](#34-ensemble-summary)
4. [Inference](#4-inference)
   - [Sentence-based inference function](#41-sentence-based-inference-function)
   - [Ensemble execution](#42-ensemble-execution)
5. [Evaluation](#5-evaluation)
   - [Strict offset-based evaluation](#51-strict-offset-based-evaluation)
   - [Overlap evaluation (IoU)](#52-overlap-evaluation-iou)

## 1. Environment and dependencies

Installation of the required packages and import of libraries.

In [ ]:
%pip install -q "transformers==4.57.6" "datasets==3.6.0" "accelerate==1.12.0" "spacy==3.8.7"
%pip install -q --use-pep517 "seqeval==1.2.2"
%pip install -q "https://github.com/explosion/spacy-models/releases/download/es_core_news_md-3.8.0/es_core_news_md-3.8.0-py3-none-any.whl"

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import hashlib
import importlib.metadata as package_metadata
import json
import math
import platform
import re
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import spacy
import torch
from datasets import load_dataset
from seqeval.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.optim import AdamW
from transformers import (
    AutoConfig, AutoModelForTokenClassification, AutoTokenizer,
    DataCollatorForTokenClassification, EarlyStoppingCallback,
    Trainer, TrainingArguments, pipeline, set_seed,
)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Settings > Accelerator before training.")
USE_BF16 = torch.cuda.is_bf16_supported()
USE_FP16 = not USE_BF16
print(f"GPU: {torch.cuda.get_device_name(0)} | BF16={USE_BF16} | FP16={USE_FP16}")

## 2. Configuration

### 2.1. Paths and dataset

Definition of dataset paths and selection of the base model. Training uses the 750 DisTEMIST training documents; external evaluation uses the 250 CardioCCC test documents.

In [ ]:
WORKING_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
DATASET_ROOT_OVERRIDE = None  # Optional path to the prepared data package.
EXPECTED_DATASET_FINGERPRINT = "3cef134a65040d6d1b12b3594cd30dab127a24ceda13187159de9429ad947965"

BASE_MODEL = "lcampillos/roberta-es-clinical-trials-ner"
BASE_MODEL_REVISION = "cfdd0455be8bd539a27883026b24aa745c871493"
SPACY_MODEL = "es_core_news_md"
SPACY_MODEL_VERSION = "3.8.0"
REFERENCE_NOTEBOOK_SHA256 = "ba93ea4c661d7c49f904d9d8dd38c915db37eed34421ce5269b4a7d689d30c1a"
PIPELINE_CODE_SHA256 = "9454bd6c73f345ce8a0c8b131a72f266cff50ca6a20ff1049d3cdc4e05f359d8"

"""Prepare and validate only DisTEMIST train and the official CardioCCC test."""

import csv
import hashlib
import json
import shutil
import urllib.request
from pathlib import Path
from zipfile import ZipFile

DATASET_ID = "distemist-cardioccc-external-v1"
SOURCE_COMMIT = "21216f4f739f17cde890c0d51bc4fa23824b50e0"
TRAIN_URL = f"https://raw.githubusercontent.com/Honricris/Trabajo_Fin_Master/{SOURCE_COMMIT}/Datasets/distemist/distemist_train.jsonl"
TRAIN_SHA256 = "f0200564e73f8665949d09ba32d055c7435dde794f47d950d43141894cf19471"
CARDIO_URL = "https://zenodo.org/records/11368861/files/multicardioner_train%2Bdev%2Btest%2Bbg%2Bmappings_240528.zip?download=1"
CARDIO_SHA256 = "3926c2cc270f62485f74dcdcf5ae47bf876535c444b887fff2e490ce72ce0aea"
CARDIO_PREFIX = "multicardioner_train+dev+test+bg+mappings_240528/track1/cardioccc_test/"


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_verified(url, destination, expected_sha256):
    destination = Path(destination)
    if destination.is_file():
        if file_sha256(destination) != expected_sha256:
            raise ValueError(f"Checksum mismatch in {destination}; the file will not be overwritten.")
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    request = urllib.request.Request(url, headers={"User-Agent": "CardioCCC-external-evaluation/1.0"})
    with urllib.request.urlopen(request, timeout=120) as response, temporary.open("wb") as stream:
        shutil.copyfileobj(response, stream)
    if file_sha256(temporary) != expected_sha256:
        raise ValueError(f"The download from {url} does not match the pinned SHA-256.")
    temporary.replace(destination)
    return destination


def validate_dataset(root, expected_fingerprint=None):
    root = Path(root)
    manifest = json.loads((root / "dataset_manifest.json").read_text(encoding="utf-8"))
    if manifest["dataset_id"] != DATASET_ID:
        raise ValueError("The manifest does not match the expected experiment.")
    fingerprint = hashlib.sha256(json.dumps(manifest["files"], sort_keys=True, separators=(",", ":")).encode()).hexdigest()
    if manifest["fingerprint"] != fingerprint or expected_fingerprint and fingerprint != expected_fingerprint:
        raise ValueError("The dataset fingerprint does not match the pinned value.")
    for relative, digest in manifest["files"].items():
        path = root / relative
        if Path(relative).is_absolute() or ".." in Path(relative).parts or not path.is_file() or file_sha256(path) != digest:
            raise ValueError(f"Missing or modified file: {relative}")

    train = [json.loads(line) for line in (root / "distemist" / "distemist_train.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
    if len(train) != 750:
        raise ValueError("Exactly 750 original training documents are required.")
    train_hashes = []
    for record in train:
        if not record["text"] or len(record["tokens"]) != len(record["ner_tags"]) or set(record["ner_tags"]) - {0, 1, 2}:
            raise ValueError("Invalid BIO training record.")
        train_hashes.append(hashlib.sha256(record["text"].encode()).hexdigest())
    if len(set(train_hashes)) != 750:
        raise ValueError("Duplicate texts found in the training set.")

    text_files = sorted((root / "cardioccc" / "test" / "texts").glob("*.txt"))
    if len(text_files) != 250:
        raise ValueError("The external test set must contain exactly 250 documents.")
    texts = {p.stem: p.read_bytes().decode("utf-8") for p in text_files}
    if set(train_hashes) & {hashlib.sha256(text.encode()).hexdigest() for text in texts.values()}:
        raise ValueError("A document is shared between training and external test sets.")
    with (root / "cardioccc" / "test" / "gold.tsv").open(encoding="utf-8", newline="") as stream:
        gold = list(csv.DictReader(stream, delimiter="\t"))
    if len(gold) != 7884 or {r["filename"] for r in gold} != set(texts):
        raise ValueError("The external gold standard does not contain the expected 250 cases and 7,884 mentions.")
    spans = set()
    for row in gold:
        start, end = int(row["start_span"]), int(row["end_span"])
        text = texts[row["filename"]]
        key = (row["filename"], row["label"], start, end)
        if row["label"] != "ENFERMEDAD" or not 0 <= start < end <= len(text) or text[start:end] != row["text"] or key in spans:
            raise ValueError(f"Invalid external annotation: {key}")
        spans.add(key)
    return manifest


def prepare_dataset(root, local_train=None, local_archive=None):
    root = Path(root)
    if (root / "dataset_manifest.json").exists():
        return validate_dataset(root)
    root.mkdir(parents=True, exist_ok=True)
    downloads = root.parent / "source_downloads"
    train_path = Path(local_train) if local_train else download_verified(TRAIN_URL, downloads / "distemist_train.jsonl", TRAIN_SHA256)
    archive_path = Path(local_archive) if local_archive else download_verified(CARDIO_URL, downloads / "multicardioner_v3.zip", CARDIO_SHA256)
    if file_sha256(train_path) != TRAIN_SHA256 or file_sha256(archive_path) != CARDIO_SHA256:
        raise ValueError("Local sources do not match the pinned versions.")
    (root / "distemist").mkdir(exist_ok=True)
    (root / "distemist" / "distemist_train.jsonl").write_bytes(train_path.read_bytes())
    texts_dir = root / "cardioccc" / "test" / "texts"
    texts_dir.mkdir(parents=True, exist_ok=True)
    with ZipFile(archive_path) as archive:
        text_names = sorted(n for n in archive.namelist() if n.startswith(CARDIO_PREFIX + "brat/") and n.endswith(".txt"))
        gold_names = [n for n in archive.namelist() if n.startswith(CARDIO_PREFIX + "tsv/") and n.endswith(".tsv")]
        if len(text_names) != 250 or len(gold_names) != 1:
            raise ValueError("The official ZIP structure differs from the expected layout.")
        for name in text_names:
            (texts_dir / Path(name).name).write_bytes(archive.read(name))
        (root / "cardioccc" / "test" / "gold.tsv").write_bytes(archive.read(gold_names[0]))
        archive_root = CARDIO_PREFIX.split("/")[0]
        (root / "LICENSE").write_bytes(archive.read(archive_root + "/LICENSE"))
    (root / "README.txt").write_text(
        "External disease evaluation: train only on DisTEMIST train (750).\n"
        "Evaluate on CardioCCC test (250; 7,884 mentions). CardioCCC dev is excluded.\n"
        "DisTEMIST: https://github.com/Honricris/Trabajo_Fin_Master/tree/" + SOURCE_COMMIT + "/Datasets/distemist\n"
        "CardioCCC: https://zenodo.org/records/11368861 (MultiCardioNER v3, CC BY 4.0).\n"
        "Lima-López et al. (2024), MultiCardioNER Corpus. DOI: 10.5281/zenodo.11368861.\n"
        "Preserve the original texts and offsets. Do not tune on the external test set.\n",
        encoding="utf-8",
    )
    files = {str(p.relative_to(root)): file_sha256(p) for p in sorted(root.rglob("*")) if p.is_file() and p.name != "dataset_manifest.json"}
    manifest = {
        "dataset_id": DATASET_ID, "schema_version": 1,
        "counts": {"distemist_train_documents": 750, "cardioccc_test_documents": 250, "cardioccc_test_mentions": 7884},
        "sources": {"distemist": {"url": TRAIN_URL, "git_commit": SOURCE_COMMIT, "sha256": TRAIN_SHA256},
                    "cardioccc": {"record": "https://zenodo.org/records/11368861", "url": CARDIO_URL, "version": "v3", "archive_sha256": CARDIO_SHA256}},
        "files": files,
        "fingerprint": hashlib.sha256(json.dumps(files, sort_keys=True, separators=(",", ":")).encode()).hexdigest(),
    }
    (root / "dataset_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, sort_keys=True, indent=2) + "\n", encoding="utf-8")
    return validate_dataset(root)

if DATASET_ROOT_OVERRIDE is not None:
    DATASET_ROOT = Path(DATASET_ROOT_OVERRIDE)
else:
    candidates = []
    input_root = Path("/kaggle/input")
    for path in sorted(input_root.rglob("dataset_manifest.json")) if input_root.exists() else []:
        try:
            candidate = json.loads(path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, UnicodeDecodeError):
            continue
        if candidate.get("dataset_id") == DATASET_ID:
            candidates.append(path.parent)
    archives = sorted(input_root.rglob("distemist_cardioccc_datos.zip")) if input_root.exists() else []
    if len(candidates) > 1 or len(archives) > 1:
        raise RuntimeError("Multiple input packages found. Set DATASET_ROOT_OVERRIDE.")
    if candidates:
        DATASET_ROOT = candidates[0]
    elif archives:
        DATASET_ROOT = WORKING_ROOT / DATASET_ID
        with ZipFile(archives[0]) as archive:
            for member in archive.infolist():
                name = Path(member.filename)
                if name.is_absolute() or ".." in name.parts or not name.parts:
                    raise ValueError(f"Unsafe path in input archive: {member.filename}")
            archive.extractall(DATASET_ROOT)
    else:
        DATASET_ROOT = WORKING_ROOT / DATASET_ID
        print("No input package attached: downloading both public sources with pinned SHA-256 hashes.")
        prepare_dataset(DATASET_ROOT)
dataset_manifest = validate_dataset(DATASET_ROOT, EXPECTED_DATASET_FINGERPRINT)
DATA_PATHS = {
    "train_jsonl": str(DATASET_ROOT / "distemist" / "distemist_train.jsonl"),
    "text_files_test_dir": str(DATASET_ROOT / "cardioccc" / "test" / "texts"),
    "gs_mentions_tsv": str(DATASET_ROOT / "cardioccc" / "test" / "gold.tsv"),
}
print(json.dumps(dataset_manifest["counts"], indent=2))
print(f"Verified fingerprint: {dataset_manifest['fingerprint']}")

### 2.2. Labels and data loading

BIO label mapping and loading of the training JSONL file.

In [ ]:
id2label = {0: "B-ENFERMEDAD", 1: "I-ENFERMEDAD", 2: "O"}
label2id = {v: k for k, v in id2label.items()}
label_list = [id2label[i] for i in range(len(id2label))]

nlp_spacy = spacy.load(SPACY_MODEL)
if nlp_spacy.meta["version"] != SPACY_MODEL_VERSION:
    raise ValueError("The es_core_news_md version does not match the pinned version.")
train_full = load_dataset("json", data_files=DATA_PATHS["train_jsonl"], split="train")
assert len(train_full) == 750
train_record_ids = [hashlib.sha256(row["text"].encode("utf-8")).hexdigest() for row in train_full]
assert len(set(train_record_ids)) == 750
print(f"Training: {len(train_full)} DisTEMIST documents | External test: 250 CardioCCC documents")
print(f"spaCy: {nlp_spacy.meta['name']} v{nlp_spacy.meta['version']}")

### 2.3. Segmentation and label alignment

Sentence segmentation functions using spaCy and BIO label alignment during tokenization.

In [ ]:
def split_by_sentences(text, tokens, labels, nlp_spacy):
    """Split document tokens and labels into sentence segments using spaCy."""
    doc = nlp_spacy(text)
    sentences = list(doc.sents)

    if len(sentences) <= 1:
        return [(tokens, labels)]

    token_char_starts = []
    search_pos = 0
    for tok in tokens:
        idx = text.find(tok, search_pos)
        if idx == -1:
            return [(tokens, labels)]
        token_char_starts.append(idx)
        search_pos = idx + len(tok)

    results = []
    for sent in sentences:
        sent_start = sent.start_char
        sent_end = sent.end_char
        sent_token_indices = [
            i for i, cs in enumerate(token_char_starts)
            if sent_start <= cs < sent_end
        ]
        if not sent_token_indices:
            continue
        sent_tokens = [tokens[i] for i in sent_token_indices]
        sent_labels = [labels[i] for i in sent_token_indices]
        results.append((sent_tokens, sent_labels))

    return results if results else [(tokens, labels)]


def tokenize_and_align_labels(examples, tok, nlp_spacy, max_length=512):
    """Tokenize sentences with truncation=True and propagate B->I labels to subtokens."""
    all_input_ids = []
    all_attention_masks = []
    all_labels = []
    original_lengths = []

    for doc_idx in range(len(examples["tokens"])):
        text = examples["text"][doc_idx]
        tokens = examples["tokens"][doc_idx]
        ner_tags = examples["ner_tags"][doc_idx]

        sent_chunks = split_by_sentences(text, tokens, ner_tags, nlp_spacy)

        for sent_tokens, sent_labels in sent_chunks:
            original_lengths.append(len(tok(sent_tokens, is_split_into_words=True, truncation=False, add_special_tokens=True)['input_ids']))
            tokenized = tok(
                [sent_tokens],
                is_split_into_words=True,
                truncation=True,
                max_length=max_length,
                padding=False,
            )

            word_ids = tokenized.word_ids(batch_index=0)
            previous_word_idx = None
            label_ids = []

            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != previous_word_idx:
                    label_ids.append(sent_labels[word_idx])
                else:
                    prev_label = sent_labels[word_idx]
                    label_ids.append(1 if prev_label == 0 else prev_label)
                previous_word_idx = word_idx

            all_input_ids.append(tokenized["input_ids"][0])
            all_attention_masks.append(tokenized["attention_mask"][0])
            all_labels.append(label_ids)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_labels,
        "original_token_length": original_lengths,
    }

### 2.4. Hyperparameters and tokenizer

Experiment configuration: training hyperparameters and tokenizer loading.

In [ ]:
BASE_MODEL_TAG = BASE_MODEL.split("/")[-1]
MAX_EPOCHS = 20
BATCH_SIZE = 16
LEARNING_RATE = 8.516e-5
LR_LAYER_DECAY = 0.95
LR_ENCODER_GROUPS = 3
DROPOUT = 0.1
WEIGHT_DECAY = 0.1844
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_THRESHOLD = 1e-4
K_FOLDS = 5
CV_SPLIT_SEED = 42
SEEDS = [123, 4242]
ENSEMBLE_VOTING_RATIO = 0.5
MAX_INPUT_TOKENS = 512
BOOTSTRAP_SAMPLES = 10000
BOOTSTRAP_SEED = 20261004

RESULTS_DIR = WORKING_ROOT / f"results_{BASE_MODEL_TAG}_cardioccc_external_kfold_multiseed"
MODEL_OUTPUT_PREFIX = f"{BASE_MODEL_TAG}-distemist-ner"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

config = AutoConfig.from_pretrained(
    BASE_MODEL, revision=BASE_MODEL_REVISION,
    num_labels=len(label2id), label2id=label2id, id2label=id2label,
    hidden_dropout_prob=DROPOUT, attention_probs_dropout_prob=DROPOUT,
    classifier_dropout=DROPOUT, attn_implementation="sdpa",
)
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL, revision=BASE_MODEL_REVISION, add_prefix_space=True,
    do_lower_case=False, keep_accents=True,
    model_max_length=config.max_position_embeddings,
)
hyperparams = {
    "base_model": BASE_MODEL, "base_model_revision": BASE_MODEL_REVISION,
    "tokenizer_revision": BASE_MODEL_REVISION,
    "max_epochs": MAX_EPOCHS, "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE, "lr_layer_decay": LR_LAYER_DECAY,
    "lr_encoder_groups": LR_ENCODER_GROUPS, "dropout": DROPOUT,
    "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_threshold": EARLY_STOPPING_THRESHOLD,
    "k_folds": K_FOLDS, "cv_split_seed": CV_SPLIT_SEED, "seeds": SEEDS,
    "ensemble_voting_ratio": ENSEMBLE_VOTING_RATIO,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "segmentation": "individual_spacy_sentences",
    "spacy_model": SPACY_MODEL, "spacy_model_version": SPACY_MODEL_VERSION,
    "bf16": USE_BF16, "fp16": USE_FP16,
    "training_corpus": "DisTEMIST train only: 750 documents",
    "evaluation_corpus": "CardioCCC official test only: 250 documents",
    "bootstrap_samples": BOOTSTRAP_SAMPLES, "bootstrap_seed": BOOTSTRAP_SEED,
}
software = {name: package_metadata.version(name) for name in
            ["torch", "transformers", "datasets", "accelerate", "seqeval", "spacy", "numpy", "pandas", "tokenizers", "huggingface-hub"]}
protocol = {"hyperparameters": hyperparams, "dataset_fingerprint": dataset_manifest["fingerprint"],
            "software": software, "reference_notebook_sha256": REFERENCE_NOTEBOOK_SHA256,
            "pipeline_code_sha256": PIPELINE_CODE_SHA256}
PROTOCOL_SHA256 = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode()).hexdigest()
manifest_path = RESULTS_DIR / "run_manifest.json"
if manifest_path.exists():
    previous = json.loads(manifest_path.read_text())
    if previous["protocol_sha256"] != PROTOCOL_SHA256:
        raise RuntimeError("This directory contains a different configuration. Use a new output directory.")
run_manifest = {**protocol, "protocol_sha256": PROTOCOL_SHA256,
                "python": platform.python_version(), "cuda": torch.version.cuda,
                "training_gpu": torch.cuda.get_device_name(0), "inference_device": "cuda:0",
                "data_paths": DATA_PATHS, "new_training_run": True}
manifest_path.write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(RESULTS_DIR / "hyperparameters.json").write_text(json.dumps(hyperparams, indent=2) + "\n", encoding="utf-8")
(RESULTS_DIR / "dataset_manifest.json").write_text(json.dumps(dataset_manifest, indent=2) + "\n", encoding="utf-8")
print(f"Fixed protocol: {PROTOCOL_SHA256}")
print(f"Results and final weights: {RESULTS_DIR}")
print("Precision specified before evaluation: BF16 when supported; FP16 otherwise.")

## 3. Training

### 3.1. Evaluation metrics

Definition of the seqeval metrics for NER evaluation during training.

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [[label_list[int(pred)] for pred, label in zip(sequence, gold) if label != -100]
                        for sequence, gold in zip(predictions, labels)]
    true_labels = [[label_list[int(label)] for label in gold if label != -100] for gold in labels]
    return {
        "precision": precision_score(true_labels, true_predictions, zero_division=0),
        "recall": recall_score(true_labels, true_predictions, zero_division=0),
        "f1": f1_score(true_labels, true_predictions, zero_division=0),
        "accuracy": accuracy_score(true_labels, true_predictions),
    }

### 3.2. Discriminative fine-tuning

Assignment of different learning rates according to layer depth.

In [ ]:
def create_discriminative_optimizer(model):
    named_params = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    layer_re = re.compile(r"\.(?:encoder\.layer|layer|layers|block|h)\.(\d+)\.")
    layer_ids = [int(m.group(1)) for n, _ in named_params for m in [layer_re.search(n.lower())] if m]
    max_layer_id = max(layer_ids) if layer_ids else 0

    groups = {}
    for name, param in named_params:
        lname = name.lower()
        if "classifier" in lname or "crf" in lname:
            bucket, lr = "head", LEARNING_RATE
        elif "embed" in lname:
            bucket, lr = "embeddings", LEARNING_RATE * (LR_LAYER_DECAY ** (LR_ENCODER_GROUPS + 1))
        else:
            match = layer_re.search(lname)
            if match and max_layer_id > 0:
                zone = min(int((int(match.group(1)) / max_layer_id) * LR_ENCODER_GROUPS), LR_ENCODER_GROUPS - 1)
                bucket, lr = f"encoder_{zone}", LEARNING_RATE * (LR_LAYER_DECAY ** (LR_ENCODER_GROUPS - zone))
            else:
                bucket, lr = "head", LEARNING_RATE

        if bucket not in groups:
            groups[bucket] = {"params": [], "lr": float(lr), "param_count": 0, "tensor_count": 0}
        groups[bucket]["params"].append(param)
        groups[bucket]["param_count"] += param.numel()
        groups[bucket]["tensor_count"] += 1

    optimizer = AdamW(
        [{"params": g["params"], "lr": g["lr"], "weight_decay": WEIGHT_DECAY} for g in groups.values()],
        lr=LEARNING_RATE,
        fused=torch.cuda.is_available(),
    )
    order = ["embeddings"] + [f"encoder_{i}" for i in range(LR_ENCODER_GROUPS)] + ["head"]
    summary = [
        {"bucket": b, "lr": groups[b]["lr"], "param_count": groups[b]["param_count"], "tensor_count": groups[b]["tensor_count"]}
        for b in order if b in groups
    ]
    return optimizer, summary

### 3.3. Multi-seed k-fold loop

Training across document folds and seeds with early stopping on internal validation. The resulting models form the ensemble. CardioCCC is reserved for external evaluation.

In [ ]:
def make_kfold_indices(n_samples, k_folds, split_seed):
    rng = np.random.default_rng(split_seed)
    indices = np.arange(n_samples)
    rng.shuffle(indices)
    fold_sizes = np.full(k_folds, n_samples // k_folds, dtype=int)
    fold_sizes[: n_samples % k_folds] += 1
    folds, current = [], 0
    for size in fold_sizes:
        val_idx = indices[current:current + size]
        train_idx = np.concatenate((indices[:current], indices[current + size:]))
        folds.append((train_idx, val_idx))
        current += size
    return folds


def verify_saved_member(member_path):
    record = json.loads(member_path.read_text(encoding="utf-8"))
    if record["protocol_sha256"] != PROTOCOL_SHA256:
        raise RuntimeError(f"Incompatible protocol in {member_path}")
    model_dir = RESULTS_DIR / record["model_dir"]
    for relative, digest in record["files"].items():
        if file_sha256(model_dir / relative) != digest:
            raise RuntimeError(f"Incomplete or modified model: {model_dir / relative}")
    if not (model_dir / "model.safetensors").is_file():
        raise RuntimeError(f"Final weights not found in {model_dir}")
    return record


def persist_ensemble_progress(records):
    (RESULTS_DIR / "ensemble_fold_seed_summary.json").write_text(json.dumps(records, indent=2) + "\n", encoding="utf-8")
    pd.DataFrame([{k: v for k, v in row.items() if k != "files"} for row in records]).to_csv(
        RESULTS_DIR / "ensemble_fold_seed_summary.csv", index=False)


fold_seed_results = []
ensemble_models = []
folds = make_kfold_indices(len(train_full), K_FOLDS, CV_SPLIT_SEED)
fold_manifest = []
print(f"Training | docs={len(train_full)} | folds={K_FOLDS} | seeds={SEEDS}")

for fold_idx, (train_idx, val_idx) in enumerate(folds, start=1):
    assert len(train_idx) == 600 and len(val_idx) == 150
    assert not set(train_idx) & set(val_idx)
    fold_manifest.append({"fold": fold_idx,
                          "train_record_sha256": [train_record_ids[i] for i in train_idx],
                          "validation_record_sha256": [train_record_ids[i] for i in val_idx]})
    train_raw = train_full.select(train_idx.tolist())
    val_raw = train_full.select(val_idx.tolist())
    fn = lambda x: tokenize_and_align_labels(x, tokenizer, nlp_spacy, max_length=MAX_INPUT_TOKENS)
    kwargs = dict(batched=True, remove_columns=train_full.column_names)
    train_ds = train_raw.map(fn, **kwargs)
    val_ds = val_raw.map(fn, **kwargs)
    lengths = {"train": np.asarray(train_ds["original_token_length"]),
               "validation": np.asarray(val_ds["original_token_length"])}
    diagnostics = {split: {"segments": len(values), "maximum_tokens": int(values.max()),
                           "truncated_segments": int((values > MAX_INPUT_TOKENS).sum())}
                   for split, values in lengths.items()}
    (RESULTS_DIR / f"tokenization_fold{fold_idx}.json").write_text(json.dumps(diagnostics, indent=2) + "\n")
    train_ds = train_ds.remove_columns(["original_token_length"])
    val_ds = val_ds.remove_columns(["original_token_length"])
    print(f"Fold {fold_idx}: train={len(train_raw)} / val={len(val_raw)} docs | {diagnostics}")

    for seed in SEEDS:
        output_dir = RESULTS_DIR / f"{MODEL_OUTPUT_PREFIX}-fold{fold_idx}-seed{seed}"
        final_dir = output_dir / "final_model"
        member_path = output_dir / "member_manifest.json"
        if member_path.is_file():
            record = verify_saved_member(member_path)
            fold_seed_results.append(record)
            ensemble_models.append(record)
            print(f"Reusing verified ensemble member: fold={fold_idx}, seed={seed}")
            continue

        set_seed(seed)
        start_time = time.time()
        model = AutoModelForTokenClassification.from_pretrained(
            BASE_MODEL, revision=BASE_MODEL_REVISION, config=config, ignore_mismatched_sizes=True)
        model.gradient_checkpointing_enable()
        optimizer, lr_summary = create_discriminative_optimizer(model)
        training_args = TrainingArguments(
            output_dir=str(output_dir), eval_strategy="epoch", logging_strategy="epoch",
            save_strategy="epoch", num_train_epochs=MAX_EPOCHS, load_best_model_at_end=True,
            metric_for_best_model="eval_f1", greater_is_better=True,
            save_total_limit=1, save_safetensors=True, save_only_model=True,
            learning_rate=LEARNING_RATE, warmup_ratio=WARMUP_RATIO, weight_decay=WEIGHT_DECAY,
            per_device_train_batch_size=BATCH_SIZE, dataloader_num_workers=2,
            dataloader_prefetch_factor=4, dataloader_persistent_workers=True,
            seed=seed, bf16=USE_BF16, fp16=USE_FP16, report_to="none",
        )
        trainer = Trainer(
            model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds,
            processing_class=tokenizer, compute_metrics=compute_metrics,
            data_collator=DataCollatorForTokenClassification(tokenizer),
            callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE,
                                            early_stopping_threshold=EARLY_STOPPING_THRESHOLD)],
            optimizers=(optimizer, None),
        )
        trainer.train()
        val_metrics = trainer.evaluate(val_ds)
        trainer.save_model(str(final_dir))
        tokenizer.save_pretrained(str(final_dir))
        trainer.state.save_to_json(str(final_dir / "trainer_state.json"))
        weights = final_dir / "model.safetensors"
        if not weights.is_file() or weights.stat().st_size == 0:
            raise RuntimeError(f"Final weights were not saved: {weights}")
        files = {str(p.relative_to(final_dir)): file_sha256(p) for p in sorted(final_dir.rglob("*")) if p.is_file()}
        record = {
            "fold": fold_idx, "seed": seed, "protocol_sha256": PROTOCOL_SHA256,
            "model_dir": str(final_dir.relative_to(RESULTS_DIR)), "files": files,
            "best_training_checkpoint": trainer.state.best_model_checkpoint,
            "train_docs": len(train_raw), "val_docs": len(val_raw),
            "train_sequences": len(train_ds), "val_sequences": len(val_ds),
            "best_eval_f1": float(trainer.state.best_metric),
            **{k: float(v) for k, v in val_metrics.items() if k.startswith("eval_")},
            "elapsed_min": (time.time() - start_time) / 60,
            "optimizer_groups": lr_summary,
        }
        member_path.write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
        verify_saved_member(member_path)
        fold_seed_results.append(record)
        ensemble_models.append(record)
        persist_ensemble_progress(fold_seed_results)
        print(f"Saved fold={fold_idx}, seed={seed} | val F1={record['eval_f1']:.4f} | {record['elapsed_min']:.1f} min")
        del trainer, model, optimizer
        gc.collect()
        torch.cuda.empty_cache()

(RESULTS_DIR / "fold_document_manifest.json").write_text(json.dumps(fold_manifest, indent=2) + "\n")
assert {(r["fold"], r["seed"]) for r in ensemble_models} == {(f, s) for f in range(1, 6) for s in SEEDS}
assert len(ensemble_models) == 10
persist_ensemble_progress(fold_seed_results)
df_ensemble_results = pd.DataFrame(fold_seed_results).sort_values(["eval_f1", "fold", "seed"], ascending=[False, True, True]).reset_index(drop=True)
print(df_ensemble_results[["fold", "seed", "eval_f1", "best_eval_f1", "elapsed_min"]].to_string(index=False))

### 3.4. Ensemble summary

Aggregation of validation metrics across folds and seeds, and saving of the ensemble state.

In [ ]:
cols = ["eval_precision", "eval_recall", "eval_f1", "eval_accuracy", "eval_loss"]
agg = df_ensemble_results[cols].agg(["mean", "std", "min", "max"]).T.reset_index().rename(columns={"index": "metric"})
print(agg.to_string(index=False))
summary = {c: {"mean": float(df_ensemble_results[c].mean()), "std": float(df_ensemble_results[c].std(ddof=0))} for c in cols}
summary["ensemble_size"] = len(ensemble_models)
(RESULTS_DIR / "validation_ensemble_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print("All ten final models are saved in each member's final_model/ directory.")

## 4. Inference

### 4.1. Sentence-based inference function

Segment each document using spaCy, apply the NER pipeline to each sentence and adjust offsets to the full text.

In [ ]:
def sentence_based_ner(texto, pipeline_ner, nlp_spacy, tokenizer_inf, diagnostics):
    doc = nlp_spacy(texto)
    all_entities = []
    for sent in doc.sents:
        sentence = texto[sent.start_char:sent.end_char]
        token_count = len(tokenizer_inf(sentence, truncation=False, add_special_tokens=True)["input_ids"])
        diagnostics["segments"] += 1
        diagnostics["maximum_tokens"] = max(diagnostics["maximum_tokens"], token_count)
        diagnostics["truncated_segments"] += int(token_count > MAX_INPUT_TOKENS)
        for entity in pipeline_ner(sentence):
            if entity["entity_group"] != "ENFERMEDAD":
                continue
            start, end = int(entity["start"]) + sent.start_char, int(entity["end"]) + sent.start_char
            if not sent.start_char <= start < end <= sent.end_char:
                raise ValueError("Inference offsets fall outside the sentence segment.")
            all_entities.append((start, end))
    return sorted(set(all_entities))


def aggregate_member_votes(member_predictions, expected_members=10):
    if len(member_predictions) != expected_members:
        raise ValueError("The ensemble requires all ten complete members.")
    votes = defaultdict(int)
    for predictions in member_predictions:
        for key in set(predictions):
            votes[key] += 1
    required = math.ceil(expected_members * ENSEMBLE_VOTING_RATIO)
    return sorted(key for key, count in votes.items() if count >= required)

### 4.2. Ensemble execution

Read the test texts, run inference with each ensemble model and aggregate entities by majority voting.

In [ ]:
ruta_txts = Path(DATA_PATHS["text_files_test_dir"])
texts_by_filename = {p.stem: p.read_bytes().decode("utf-8") for p in sorted(ruta_txts.glob("*.txt"))}
assert len(texts_by_filename) == 250
assert len(ensemble_models) == 10
pred_file = RESULTS_DIR / "predictions_cardioccc_ensemble_k5_s2.tsv"
member_dir = RESULTS_DIR / "member_predictions"
member_dir.mkdir(exist_ok=True)
member_predictions, model_times, coverage = [], [], []
end_to_end_start = time.time()
print("External test: 250 documents | Models: 10 | Required votes: 5")

for member in ensemble_models:
    model_dir = RESULTS_DIR / member["model_dir"]
    verify_saved_member(model_dir.parent / "member_manifest.json")
    started = time.time()
    modelo_inf = AutoModelForTokenClassification.from_pretrained(str(model_dir), attn_implementation="sdpa")
    tokenizer_inf = AutoTokenizer.from_pretrained(str(model_dir), model_max_length=MAX_INPUT_TOKENS)
    nlp_ner = pipeline("ner", model=modelo_inf, tokenizer=tokenizer_inf,
                       aggregation_strategy="simple", device=0)
    predictions, rows = [], []
    diagnostics = {"segments": 0, "maximum_tokens": 0, "truncated_segments": 0}
    for filename, texto in texts_by_filename.items():
        entities = sentence_based_ner(texto, nlp_ner, nlp_spacy, tokenizer_inf, diagnostics)
        coverage.append({"filename": filename, "fold": member["fold"], "seed": member["seed"],
                         "completed": True, "predicted_mentions": len(entities)})
        for start, end in entities:
            predictions.append((filename, "ENFERMEDAD", start, end))
            rows.append({"filename": filename, "label": "ENFERMEDAD", "off0": start, "off1": end, "span": texto[start:end]})
    elapsed = time.time() - started
    model_times.append(elapsed)
    member_predictions.append(predictions)
    stem = f"fold{member['fold']}_seed{member['seed']}"
    pd.DataFrame(rows, columns=["filename", "label", "off0", "off1", "span"]).to_csv(member_dir / f"{stem}.tsv", sep="\t", index=False)
    (member_dir / f"{stem}_diagnostics.json").write_text(json.dumps(diagnostics, indent=2) + "\n")
    print(f"{stem}: {elapsed:.1f}s | {len(rows)} mentions | {diagnostics}")
    del nlp_ner, tokenizer_inf, modelo_inf
    gc.collect()
    torch.cuda.empty_cache()

consensus = aggregate_member_votes(member_predictions)
counters, final_rows = defaultdict(int), []
for filename, label, start, end in consensus:
    counters[filename] += 1
    final_rows.append({"filename": filename, "mark": f"T{counters[filename]}", "label": label,
                       "off0": start, "off1": end, "span": texts_by_filename[filename][start:end]})
df_pred = pd.DataFrame(final_rows, columns=["filename", "mark", "label", "off0", "off1", "span"])
df_pred.to_csv(pred_file, sep="\t", index=False)
pd.DataFrame(coverage).to_csv(RESULTS_DIR / "document_processing_manifest.csv", index=False)
assert len(coverage) == 2500 and all(row["completed"] for row in coverage)
inference_seconds = time.time() - end_to_end_start
stats = {"documents": 250, "ensemble_members": 10, "required_votes": 5,
         "predicted_mentions": len(df_pred), "inference_total_seconds": inference_seconds,
         "inference_avg_file_seconds": inference_seconds / 250,
         "member_seconds": model_times, "device": "cuda:0",
         "prediction_empty_documents": [name for name in texts_by_filename if not counters[name]]}
(RESULTS_DIR / "inference_stats_ensemble.json").write_text(json.dumps(stats, indent=2) + "\n")
print(f"Predictions saved: {pred_file} | Complete ensemble: {inference_seconds:.1f}s")

## 5. Evaluation

### 5.1. Strict offset-based evaluation

Compare predictions against the gold standard using exact label and character-offset matches across all 250 documents. Estimate the F1 confidence interval by document bootstrap.

In [ ]:
def prf(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return precision, recall, f1


def strict_document_counts(gold, predicted, document_ids):
    documents = sorted(document_ids)
    if {key[0] for key in gold | predicted} - set(documents):
        raise ValueError("Evaluation contains document IDs outside the fixed test set.")
    rows = []
    for name in documents:
        gs = {key for key in gold if key[0] == name}
        ps = {key for key in predicted if key[0] == name}
        rows.append({"filename": name, "tp": len(gs & ps), "fp": len(ps - gs), "fn": len(gs - ps)})
    return rows


df_gs = pd.read_csv(DATA_PATHS["gs_mentions_tsv"], sep="\t").rename(
    columns={"start_span": "off0", "end_span": "off1", "text": "span"})
df_pred = pd.read_csv(pred_file, sep="\t")
set_gs = set(zip(df_gs["filename"], df_gs["label"], df_gs["off0"], df_gs["off1"]))
set_pred = set(zip(df_pred["filename"], df_pred["label"], df_pred["off0"], df_pred["off1"]))
assert len(df_gs) == len(set_gs) == 7884
per_document = strict_document_counts(set_gs, set_pred, texts_by_filename)
per_document_df = pd.DataFrame(per_document)
per_document_df.to_csv(RESULTS_DIR / "strict_counts_per_document.tsv", sep="\t", index=False)
tp, fp, fn = [int(per_document_df[column].sum()) for column in ["tp", "fp", "fn"]]
precision, recall, fscore = prf(tp, fp, fn)
strict_report = {"base_model": BASE_MODEL, "model_revision": BASE_MODEL_REVISION,
                 "protocol_sha256": PROTOCOL_SHA256, "documents": 250,
                 "gold_mentions": len(set_gs), "ensemble_size": 10, "required_votes": 5,
                 "tp": tp, "fp": fp, "fn": fn, "precision": precision,
                 "recall": recall, "fscore": fscore,
                 "inference_total_seconds": inference_seconds,
                 "predictions_file": str(pred_file.relative_to(RESULTS_DIR))}
(RESULTS_DIR / "strict_evaluation_ensemble.json").write_text(json.dumps(strict_report, indent=2) + "\n")
print(f"Complete CardioCCC test | P={precision:.4f} | R={recall:.4f} | F1={fscore:.4f}")
print(f"TP={tp} FP={fp} FN={fn} | Gold={len(set_gs)} | Documents=250")

def bootstrap_micro_f1(counts, samples=10000, seed=20261004):
    counts = np.asarray(counts, dtype=np.int64)
    if counts.ndim != 2 or counts.shape[1] != 3 or len(counts) == 0 or np.any(counts < 0):
        raise ValueError("Non-negative TP/FP/FN counts per document are required.")
    rng = np.random.default_rng(seed)
    values = np.empty(samples)
    for i in range(samples):
        draw = rng.integers(0, len(counts), size=len(counts))
        tp, fp, fn = counts[draw].sum(axis=0)
        values[i] = prf(int(tp), int(fp), int(fn))[2]
    return np.percentile(values, [2.5, 97.5]).tolist()

ci = bootstrap_micro_f1(per_document_df[["tp", "fp", "fn"]].to_numpy(), BOOTSTRAP_SAMPLES, BOOTSTRAP_SEED)
bootstrap_report = {"system": "lcampillos ensemble", "fscore": fscore,
                    "percentile_95_ci": ci, "samples": BOOTSTRAP_SAMPLES,
                    "seed": BOOTSTRAP_SEED, "document_order": per_document_df["filename"].tolist(),
                    "conditional_on_fitted_system": True, "paired_ministral_comparison": "pending"}
(RESULTS_DIR / "bootstrap_summary.json").write_text(json.dumps(bootstrap_report, indent=2) + "\n")
print(f"F1={fscore:.4f} | 95% percentile CI [{ci[0]:.4f}, {ci[1]:.4f}]")

### 5.2. Overlap evaluation (IoU)

Calculate precision, recall and F1 at different overlap thresholds between predicted and gold spans.

In [ ]:
thresholds = [0.0, 0.5, 0.8]
results = {threshold: {"tp": 0, "fp": 0, "fn": 0} for threshold in thresholds}
for filename in sorted(texts_by_filename):
    gold_intervals = sorted({(a, b) for name, label, a, b in set_gs if name == filename})
    pred_intervals = sorted({(a, b) for name, label, a, b in set_pred if name == filename})
    candidates = []
    for pi, (p0, p1) in enumerate(pred_intervals):
        for gi, (g0, g1) in enumerate(gold_intervals):
            intersection = min(p1, g1) - max(p0, g0)
            if intersection > 0:
                candidates.append((intersection / (max(p1, g1) - min(p0, g0)), pi, gi))
    candidates.sort(reverse=True)
    for threshold in thresholds:
        matched_p, matched_g = set(), set()
        for iou, pi, gi in candidates:
            if iou >= threshold and pi not in matched_p and gi not in matched_g:
                matched_p.add(pi)
                matched_g.add(gi)
        results[threshold]["tp"] += len(matched_p)
        results[threshold]["fp"] += len(pred_intervals) - len(matched_p)
        results[threshold]["fn"] += len(gold_intervals) - len(matched_g)
overlap_report = {}
for threshold, counts in results.items():
    p, r, f1 = prf(counts["tp"], counts["fp"], counts["fn"])
    overlap_report[str(threshold)] = {**counts, "precision": p, "recall": r, "fscore": f1}
    print(f"IoU >= {threshold}: P={p:.4f} R={r:.4f} F1={f1:.4f}")
(RESULTS_DIR / "overlap_eval_summary.json").write_text(json.dumps(overlap_report, indent=2) + "\n")

artifact_files = {str(p.relative_to(RESULTS_DIR)): file_sha256(p)
                  for p in sorted(RESULTS_DIR.rglob("*"))
                  if p.is_file() and p.name != "artifact_checksums.json"}
(RESULTS_DIR / "artifact_checksums.json").write_text(json.dumps(artifact_files, indent=2) + "\n")
final_weight_files = sorted(RESULTS_DIR.glob("*/final_model/model.safetensors"))
assert len(final_weight_files) == 10
print(f"Final weights retained: {len(final_weight_files)} models")
print(f"Total result files: {len(artifact_files)}")
print(f"Directory to retain or download from Kaggle: {RESULTS_DIR}")